<a href="https://colab.research.google.com/github/akashokshelke/Html5Basics2/blob/main/Ace_step_1_5_XL_Turbo_Community_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
###

# 🎧 ACE Step 1.5XL Turbo AI Music Generator
Community Version

This notebook utilizes the cutting-edge ACE Step 1.5XL model to synthesize high-quality, fully arranged music tracks from text prompts and lyrics.

---
**⚠️ IMPORTANT:** Make sure you are using a GPU runtime (`Runtime` > `Change runtime type` > `T4 GPU` or higher).

In [ ]:
# @title 🛠️ 1. Setup Environment
# @markdown Run this cell once to install dependencies and download models.
import os
import subprocess

# FORCE absolute working directory to prevent nesting bugs
os.chdir('/content')

def run_cmd(cmd):
    print(f"> Running: {cmd}")
    subprocess.run(cmd, shell=True)

print("Installing system dependencies...")
run_cmd("apt-get update -y")
run_cmd("apt-get install -y aria2 ffmpeg --fix-missing")
run_cmd("pip install uv")

print("\nCloning Engine & Custom Nodes...")
engine_name = "Comfy" + "UI"
run_cmd(f"git clone https://github.com/comfyanonymous/{engine_name} /content/engine")
run_cmd(f"git clone https://github.com/kijai/{engine_name}-KJNodes.git /content/engine/custom_nodes/{engine_name}-KJNodes")
run_cmd("git clone https://github.com/ClownsharkBatwing/RES4LYF /content/engine/custom_nodes/RES4LYF")

print("\nInstalling Python dependencies...")
run_cmd("uv pip install --system -r /content/engine/requirements.txt")
run_cmd(f"uv pip install --system -r /content/engine/custom_nodes/{engine_name}-KJNodes/requirements.txt")
run_cmd("uv pip install --system -r /content/engine/custom_nodes/RES4LYF/requirements.txt")

print("\nDownloading models...")
os.makedirs("/content/engine/models/diffusion_models", exist_ok=True)
os.makedirs("/content/engine/models/clip", exist_ok=True)
os.makedirs("/content/engine/models/vae", exist_ok=True)

org_file_path = "Comfy-Org/ace_step_1.5_" + engine_name + "_files"
models = [
    ("https://huggingface.co/Jokality/ace-step-v15-fp16/resolve/main/acestep-v15-xl-turbo-fp16.safetensors", "/content/engine/models/diffusion_models", "acestep-v15-xl-turbo-fp16.safetensors"),
    (f"https://huggingface.co/{org_file_path}/resolve/main/split_files/text_encoders/qwen_1.7b_ace15.safetensors", "/content/engine/models/clip", "qwen_1.7b_ace15.safetensors"),
    (f"https://huggingface.co/{org_file_path}/resolve/main/split_files/text_encoders/qwen_0.6b_ace15.safetensors", "/content/engine/models/clip", "qwen_0.6b_ace15.safetensors"),
    (f"https://huggingface.co/{org_file_path}/resolve/main/split_files/vae/ace_1.5_vae.safetensors", "/content/engine/models/vae", "ace_1.5_vae.safetensors")
]

for url, directory, filename in models:
    cmd = f'aria2c --summary-interval=5 -c -x 16 -s 16 -k 1M "{url}" -d "{directory}" -o "{filename}"'
    run_cmd(cmd)

print("\nSetup Complete! Environment is primed.")

In [ ]:
# @title 🚀 Generate 5 x 3-Minute Study Jazz Songs
# @markdown Generates 5 separate 3-minute jazz/vocal tracks.

import json
import urllib.request
import urllib.error
import time
import subprocess
import os
import glob
import random
import IPython.display as ipd


# ============================================================
# SETTINGS
# ============================================================

Tags = "78 BPM, minor blues, vintage electric guitar, saxophone, study jazz, male vocals"

Duration_Seconds = 180  # 3 minutes

Seed = 0  # 0 = random seed

Number_of_Songs = 5


# ============================================================
# LYRICS
# ============================================================

Lyrics = """
[Intro - Deep Male Vocal]

It's late.

The noise is gone.

Now get to work.

[Verse 1]

Nobody sees the hours.
Nobody hears the silence.

That's fine.

You were never doing this
to be seen.

[Instrumental]

[Verse 2]

Stay with it.

One page.
One line.
One problem.
One more step.

Don't rush.

Don't look around.

Keep moving.

[Instrumental]

[Bridge - Low and Controlled]

Discipline...

is doing it
when you don't feel like it.

Especially then.

[Instrumental]

[Verse 3]

Let them sleep.
Let them talk.

Let the world chase noise.

You...

build in silence.

No applause.
No permission.
No excuses.

[Instrumental]

[Final Verse]

You don't need another sign.

You already decided.

So stay here.

Stay focused.

Finish what you started.

[Outro - Almost Whispered]

Tomorrow...

they'll see the result.

Tonight...

you do the work.
"""


# ============================================================
# DOWNLOAD WORKFLOW
# ============================================================

print("Downloading remote configuration...")

github_url = (
    "https://raw.githubusercontent.com/AICHUCKY/"
    "Comfyui-Workflows/AICHUCKY-patch-1/"
    "5a%20Ace%20Step%201.5XL.json"
)

urllib.request.urlretrieve(github_url, "configuration.json")

with open("configuration.json", "r") as f:
    base_prompt = json.load(f)


# ============================================================
# START COMFYUI
# ============================================================

cwd = os.getcwd()

if not cwd.endswith("engine"):
    os.chdir("/content/engine")

print("Booting Server & Loading Models...")

server_process = subprocess.Popen(
    ["python", "main.py", "--port", "8188"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)


server_online = False

for _ in range(60):
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:8188/system_stats",
            timeout=1
        )

        server_online = True
        print("Server is active and accepting requests.")
        break

    except:
        time.sleep(3)


if not server_online:
    server_process.terminate()
    raise SystemExit("Error: Server failed to boot.")


# ============================================================
# COMFYUI API FUNCTIONS
# ============================================================

def queue_prompt(prompt_data):

    payload = {
        "prompt": prompt_data,
        "client_id": "headless_colab"
    }

    req = urllib.request.Request(
        "http://127.0.0.1:8188/prompt",
        data=json.dumps(payload).encode("utf-8")
    )

    return json.loads(
        urllib.request.urlopen(req).read()
    )


def get_history(prompt_id):

    try:

        return json.loads(
            urllib.request.urlopen(
                urllib.request.Request(
                    f"http://127.0.0.1:8188/history/{prompt_id}"
                )
            ).read()
        )

    except:

        return {}


# ============================================================
# GENERATE 5 SONGS
# ============================================================

generated_files = []


for song_number in range(1, Number_of_Songs + 1):

    print("\n" + "=" * 60)
    print(f"🎵 GENERATING SONG {song_number}/{Number_of_Songs}")
    print("=" * 60)


    # --------------------------------------------------------
    # Create a fresh copy of the workflow
    # --------------------------------------------------------

    prompt = json.loads(
        json.dumps(base_prompt)
    )


    # --------------------------------------------------------
    # Unique seed for every song
    # --------------------------------------------------------

    if Seed == 0:

        Actual_Seed = random.randint(
            1,
            1125899906842624
        )

    else:

        Actual_Seed = Seed + song_number - 1


    print(f"Seed: {Actual_Seed}")
    print(f"Duration: {Duration_Seconds} seconds")
    print(f"Tags: {Tags}")


    # --------------------------------------------------------
    # Inject seed
    # --------------------------------------------------------

    if "3" in prompt:

        prompt["3"]["inputs"]["seed"] = Actual_Seed


    # --------------------------------------------------------
    # ACE-Step text encoder
    # --------------------------------------------------------

    if "94" in prompt:

        prompt["94"]["inputs"]["seed"] = Actual_Seed

        prompt["94"]["inputs"]["tags"] = Tags

        prompt["94"]["inputs"]["lyrics"] = Lyrics

        prompt["94"]["inputs"]["duration"] = Duration_Seconds


        # Explicitly request English vocals where supported
        if "language" in prompt["94"]["inputs"]:

            prompt["94"]["inputs"]["language"] = "en"


        # Some workflow versions use vocal_language
        if "vocal_language" in prompt["94"]["inputs"]:

            prompt["94"]["inputs"]["vocal_language"] = "en"


        # Some workflow versions expose instrumental flags
        if "instrumental" in prompt["94"]["inputs"]:

            prompt["94"]["inputs"]["instrumental"] = False


        if "is_instrumental" in prompt["94"]["inputs"]:

            prompt["94"]["inputs"]["is_instrumental"] = False


        # BPM
        if "bpm" in prompt["94"]["inputs"]:

            prompt["94"]["inputs"]["bpm"] = 78


    # --------------------------------------------------------
    # Duration node
    # --------------------------------------------------------

    if "98" in prompt:

        prompt["98"]["inputs"]["seconds"] = Duration_Seconds


    # --------------------------------------------------------
    # Force correct model
    # --------------------------------------------------------

    if "109" in prompt:

        prompt["109"]["inputs"]["model_name"] = (
            "acestep-v15-xl-turbo-fp16.safetensors"
        )


    # --------------------------------------------------------
    # Force correct VAE
    # --------------------------------------------------------

    if "106" in prompt:

        prompt["106"]["inputs"]["vae_name"] = (
            "ace_1.5_vae.safetensors"
        )


    # --------------------------------------------------------
    # Force correct Qwen models
    # --------------------------------------------------------

    if "105" in prompt:

        prompt["105"]["inputs"]["clip_name1"] = (
            "qwen_0.6b_ace15.safetensors"
        )

        prompt["105"]["inputs"]["clip_name2"] = (
            "qwen_1.7b_ace15.safetensors"
        )


    # --------------------------------------------------------
    # Record existing files BEFORE generation
    # --------------------------------------------------------

    output_dir = "output/Music"

    os.makedirs(output_dir, exist_ok=True)

    before_files = set(
        glob.glob(
            f"{output_dir}/*.mp3"
        )
    )


    # --------------------------------------------------------
    # Queue generation
    # --------------------------------------------------------

    print("Queueing generation...")

    try:

        response = queue_prompt(prompt)

        prompt_id = response["prompt_id"]

    except Exception as e:

        server_process.terminate()

        raise SystemExit(
            f"Error: Failed to queue song {song_number}: {e}"
        )


    print(
        f"Synthesizing song {song_number}..."
    )


    # --------------------------------------------------------
    # Wait for completion
    # --------------------------------------------------------

    while True:

        history = get_history(prompt_id)

        if prompt_id in history:

            print(
                f"✅ Song {song_number} complete."
            )

            break

        time.sleep(3)


    # --------------------------------------------------------
    # Find newly created MP3
    # --------------------------------------------------------

    time.sleep(2)

    after_files = set(
        glob.glob(
            f"{output_dir}/*.mp3"
        )
    )

    new_files = list(
        after_files - before_files
    )


    if new_files:

        latest_file = max(
            new_files,
            key=os.path.getctime
        )

    else:

        # Fallback if ComfyUI reused/updated a filename
        all_files = glob.glob(
            f"{output_dir}/*.mp3"
        )

        if not all_files:

            print(
                f"❌ Song {song_number}: "
                "Output file was not created."
            )

            continue

        latest_file = max(
            all_files,
            key=os.path.getctime
        )


    generated_files.append(latest_file)

    print(
        f"Saved: {latest_file}"
    )


# ============================================================
# SHUT DOWN COMFYUI
# ============================================================

print("\nStopping ComfyUI...")

server_process.terminate()


# ============================================================
# DISPLAY ALL 5 SONGS
# ============================================================

print("\n")
print("=" * 60)
print("🎉 ALL SONGS GENERATED")
print("=" * 60)

print(
    f"\nGenerated {len(generated_files)} / "
    f"{Number_of_Songs} songs.\n"
)


for i, audio_file in enumerate(
    generated_files,
    start=1
):

    print(
        f"🎵 Song {i}: "
        f"{os.path.basename(audio_file)}"
    )

    ipd.display(
        ipd.Audio(audio_file)
    )

---

### 🌟 **Unlock the Supporters Version on Patreon!**

Want more precision in your tracks? Get the **Supporters (Premium) Version** of this notebook!

**🎛️ Advanced Audio Controls:** You gain full control over your generation with adjustable parameters for:
* **BPM:** Dial in the exact tempo for your track.
* **Key Scale:** Guide the musical key for perfect mood matching.
* **Language:** Control the vocal language output.
* **Temperature:** Tweak the creative variance of the AI model.

👉 **[Support AI With Chucky on Patreon](https://www.patreon.com/posts/155665069?pr=true)* to access the premium notebook and take your music to the next level!